# Dataset Preparation and Validation

## Trustworthy AI — Watermarking Diffusion Language Models

This notebook prepares the text generations produced by
LLaDA-8B-Instruct for subsequent watermark detection and
robustness experiments.

### Experimental data

We generated 1,200 responses across three WaterBench datasets:

| Dataset | Watermarked | Unwatermarked |
|---|---:|---:|
| Longform QA | 200 | 200 |
| Finance QA | 200 | 200 |
| AlpacaFarm | 200 | 200 |
| **Total** | **600** | **600** |

### Objectives

1. Identify and inspect all generated JSONL files.
2. Detect duplicate or missing responses.
3. Remove duplicate records without modifying the originals.
4. Verify prompt alignment across experimental conditions.
5. Assemble two validated datasets of 600 responses each.

All processing is performed locally without GPU acceleration.

## 1. Data Loading

The original generation outputs are stored in the `results/`
directory.

Each JSONL file contains the generated responses for one
batch of 20 prompts under one experimental condition.

We first identify all available files before performing
any modifications.

In [2]:
from pathlib import Path
import json
import re
from collections import Counter, defaultdict

# Locate project root
ROOT = Path.cwd().resolve()

if not (ROOT / "results").is_dir():
    ROOT = ROOT.parent

RESULTS = ROOT / "results"

assert RESULTS.is_dir(), f"Results directory not found: {RESULTS}"

# Production data directories
NO_WM_DIR = RESULTS / "No watermark Data"
WM_DIR = RESULTS / "Watermark Data"

assert NO_WM_DIR.is_dir(), f"Missing directory: {NO_WM_DIR}"
assert WM_DIR.is_dir(), f"Missing directory: {WM_DIR}"

# Only production JSONL files
files = sorted(NO_WM_DIR.glob("*.jsonl")) + sorted(WM_DIR.glob("*.jsonl"))

print("Project root:", ROOT)
print("Production JSONL files:", len(files))
print("Unwatermarked files:", len(list(NO_WM_DIR.glob("*.jsonl"))))
print("Watermarked files:", len(list(WM_DIR.glob("*.jsonl"))))

Project root: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking
Production JSONL files: 60
Unwatermarked files: 30
Watermarked files: 30


## 2. Initial Dataset Inspection

Before cleaning the data, we inspect each batch to determine:

- The number of generated records.
- The number of unique prompts.
- Whether any duplicate generations are present.
- Whether all expected batches are available.

The original JSONL files remain unchanged.

In [4]:
pattern = re.compile(
    r"(2-1_longform_qa|2-2_finance_qa|5-1_alpacafarm)"
    r"_batch_(\d{2})_(watermarked|unwatermarked)"
    r"(_RAW)?\.jsonl"
)

total_records = 0
total_duplicates = 0
matched_files = 0

print(f"{'File':<53} {'Records':>8} {'Unique':>8} {'Duplicates':>11}")
print("-" * 85)

for file in files:
    if not pattern.fullmatch(file.name):
        continue

    with file.open(encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    unique_prompts = len({
        record["prompt"]
        for record in records
    })

    duplicates = len(records) - unique_prompts

    matched_files += 1
    total_records += len(records)
    total_duplicates += duplicates

    print(
        f"{file.name:<53} "
        f"{len(records):>8} "
        f"{unique_prompts:>8} "
        f"{duplicates:>11}"
    )

print("\nSummary")
print("Batch files:", matched_files)
print("Total records:", total_records)
print("Duplicate records:", total_duplicates)
print("Expected unique responses:", 1200)

File                                                   Records   Unique  Duplicates
-------------------------------------------------------------------------------------
2-1_longform_qa_batch_01_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_02_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_03_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_04_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_05_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_06_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_07_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_08_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_09_unwatermarked.jsonl                20       20           0
2-1_longform_qa_batch_10_unwatermarked.jsonl                20       20   

## 3. Duplicate Removal

Three unwatermarked AlpacaFarm batches contain duplicate generations:

- Batch 05: 40 records, 20 unique prompts
- Batch 08: 40 records, 20 unique prompts
- Batch 09: 40 records, 20 unique prompts

For each duplicated prompt, we retain the first recorded completion.

This deterministic selection rule ensures reproducibility.
The original files remain unchanged.

In [6]:
from collections import defaultdict

DATASETS = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

CONDITIONS = ["unwatermarked", "watermarked"]

# Organize the files by dataset, batch and condition
batch_files = {}

for file in files:
    match = pattern.fullmatch(file.name)

    if match:
        dataset, batch_id, condition, _ = match.groups()
        key = (dataset, int(batch_id), condition)

        assert key not in batch_files, f"Duplicate file: {key}"
        batch_files[key] = file

assert len(batch_files) == 60

cleaned = defaultdict(list)
prompts_by_batch = {}
removed_duplicates = 0

for dataset in DATASETS:
    for batch_id in range(1, 11):
        for condition in CONDITIONS:

            key = (dataset, batch_id, condition)
            path = batch_files[key]

            with path.open(encoding="utf-8") as f:
                rows = [
                    json.loads(line)
                    for line in f
                    if line.strip()
                ]

            unique = {}

            for row in rows:
                prompt = row["prompt"]

                if prompt not in unique:
                    unique[prompt] = row

            assert len(unique) == 20, (
                f"{path.name}: {len(unique)} unique prompts"
            )

            assert all(
                str(row.get("completion", "")).strip()
                for row in unique.values()
            ), f"Empty completion in {path.name}"

            duplicates = len(rows) - len(unique)
            removed_duplicates += duplicates

            if duplicates:
                print(f"{path.name}: {duplicates} duplicates removed")

            cleaned[condition].extend(unique.values())
            prompts_by_batch[key] = set(unique.keys())

# Verify matching prompts between conditions
for dataset in DATASETS:
    for batch_id in range(1, 11):

        wm = prompts_by_batch[
            (dataset, batch_id, "watermarked")
        ]

        no_wm = prompts_by_batch[
            (dataset, batch_id, "unwatermarked")
        ]

        assert wm == no_wm, (
            f"Prompt mismatch: {dataset}, batch {batch_id}"
        )

# Final checks
for condition in CONDITIONS:
    assert len(cleaned[condition]) == 600

    counts = Counter()

    for row in cleaned[condition]:
        batch_name = row["dataset_name"]

        dataset = batch_name.split("_batch_")[0]
        counts[dataset] += 1

    print(f"\n{condition}: {len(cleaned[condition])} responses")
    print("Dataset distribution:", dict(counts))

    assert all(counts.get(dataset) == 200 for dataset in DATASETS)
    assert len(counts) == 3

print(f"\nTotal duplicates removed: {removed_duplicates}")
print("All prompts are aligned between conditions.")

5-1_alpacafarm_batch_05_unwatermarked_RAW.jsonl: 20 duplicates removed
5-1_alpacafarm_batch_08_unwatermarked_RAW.jsonl: 20 duplicates removed
5-1_alpacafarm_batch_09_unwatermarked_RAW.jsonl: 20 duplicates removed

unwatermarked: 600 responses
Dataset distribution: {'2-1_longform_qa': 200, '2-2_finance_qa': 200, '5-1_alpacafarm': 200}

watermarked: 600 responses
Dataset distribution: {'2-1_longform_qa': 200, '2-2_finance_qa': 200, '5-1_alpacafarm': 200}

Total duplicates removed: 60
All prompts are aligned between conditions.


## 4. Export Cleaned Datasets

After removing duplicate records and validating prompt alignment,
we export two datasets:

- `unwatermarked_600.jsonl`: 600 responses without watermark.
- `watermarked_600.jsonl`: 600 responses with watermark.

Each dataset contains 200 responses from each WaterBench dataset.

The original batch files remain unchanged. The cleaned datasets
will be used for watermark detection and robustness experiments.

In [7]:
import json
from pathlib import Path

OUTPUT_DIR = RESULTS / "Cleaned Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for condition in CONDITIONS:
    output_path = OUTPUT_DIR / f"{condition}_600.jsonl"

    with output_path.open("w", encoding="utf-8") as f:
        for row in cleaned[condition]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Verify the exported file
    with output_path.open(encoding="utf-8") as f:
        exported = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    assert len(exported) == 600

    print(f"Created: {output_path.name}")
    print(f"Validated records: {len(exported)}")

print("\nDataset preparation completed successfully.")

Created: unwatermarked_600.jsonl
Validated records: 600
Created: watermarked_600.jsonl
Validated records: 600

Dataset preparation completed successfully.
